# Análisis de usabilidad - SUS (System Usability Scale)

Este notebook implementa el análisis de la encuesta SUS cumpliendo con los estándares de rigor metodológico:
- Cálculo de puntajes (0-100) según la fórmula estándar.
- Intervalo de Confianza (IC) al 95% utilizando la distribución t de Student.
- Escala de adjetivos cualitativos de Bangor, Kortum & Miller (2009).
- Semilla fija (SEED=42) para asegurar reproducibilidad.
- Análisis demográfico y de experiencia previa.
- Reporte explícito de amenazas a la validez.


## 1. Configuración Inicial y Semilla
Se establece la semilla para asegurar la reproducibilidad en cálculos o simulaciones, según los requisitos del estudio.

In [ ]:
import os
import sys
import csv
import numpy as np
import pandas as pd
import scipy.stats as stats
import statistics
from pathlib import Path

# Fijar semilla de reproducibilidad
SEED = 42
np.random.seed(SEED)

def raiz_repo():
    d = Path(os.path.abspath("")).resolve()
    while not (d / "Makefile").is_file() and d != d.parent:
        d = d.parent
    return d

REPO = raiz_repo()
SUS_CSV = REPO / "docs/mediciones/sus/respuestas-sus.csv"

if not SUS_CSV.exists():
    print("ESTADO: Aún no hay datos SUS (falta docs/mediciones/sus/respuestas-sus.csv).")
else:
    print("Datos SUS encontrados en:", SUS_CSV)


## 2. Reglas de Puntuación SUS y Escala de Bangor
Fórmula original de Brooke (1996):
- Ítems impares (1,3,5,7,9): `valor - 1`
- Ítems pares (2,4,6,8,10): `5 - valor`
- Suma de los 10 ítems (rango 0-40) × 2.5 = score final (rango 0-100)

Escala de Bangor, Kortum & Miller (2009):
- > 85.58: Excelente
- 71.4 a 85.58: Bueno
- 50.9 a 71.4: OK (Aceptable / Marginal)
- < 50.9: Pobre


In [ ]:
def clasificacion_bangor(score):
    if pd.isna(score):
        return "Sin puntaje"
    if score >= 85.58:
        return "Excelente"
    elif score >= 71.4:
        return "Bueno"
    elif score >= 50.9:
        return "OK (Aceptable)"
    else:
        return "Pobre"

def calcular_score_sus(row):
    # Calcula el score individual a partir de las columnas Q1 a Q10
    try:
        suma = 0
        for i in range(1, 11):
            val = int(row[f"Q{i}"])
            if i % 2 != 0:  # Impar
                suma += (val - 1)
            else:           # Par
                suma += (5 - val)
        return suma * 2.5
    except (ValueError, KeyError, TypeError):
        return np.nan


## 3. Análisis de Datos e Intervalo de Confianza (t de Student)
Dado que las muestras de usabilidad suelen ser pequeñas (n < 30), utilizamos la distribución t de Student para calcular el intervalo de confianza al 95%.

In [ ]:
if SUS_CSV.exists():
    try:
        df = pd.read_csv(SUS_CSV)
        # Si el CSV esta vacío o solo tiene headers
        if len(df) > 0:
            # Asegurar que el score este calculado
            if 'score' not in df.columns or df['score'].isnull().all():
                df['score'] = df.apply(calcular_score_sus, axis=1)

            scores = df['score'].dropna()
            n = len(scores)
            
            if n > 1:
                media = np.mean(scores)
                mediana = np.median(scores)
                desv = np.std(scores, ddof=1) # Desviacion estandar muestral
                mini = np.min(scores)
                maxi = np.max(scores)
                
                # Calculo del IC 95% usando t de Student
                t_critico = stats.t.ppf(0.975, df=n-1)
                error_estandar = desv / np.sqrt(n)
                margen_error = t_critico * error_estandar
                ic_inferior = media - margen_error
                ic_superior = media + margen_error
                
                print(f"--- RESULTADOS SUS (n={n}) ---")
                print(f"Media: {media:.2f}")
                print(f"Mediana: {mediana:.2f}")
                print(f"Desviacion Estandar: {desv:.2f}")
                print(f"Rango: {mini} - {maxi}")
                print(f"Intervalo de Confianza (95%, t-Student): [{ic_inferior:.2f}, {ic_superior:.2f}]")
                print(f"Clasificacion (Bangor): {clasificacion_bangor(media)}")
                
                # Identificando al peor puntaje y su experiencia
                peor_idx = df['score'].idxmin()
                peor_score = df.loc[peor_idx, 'score']
                peor_exp = df.loc[peor_idx, 'experiencia_web']
                print(f"\nObservacion: El participante con menor puntaje ({peor_score}) reporto experiencia web '{peor_exp}'.")
                
                print("\nDistribucion Demografica:")
                print(df[['edad', 'sexo', 'experiencia_web', 'dispositivo']].value_counts().to_string())
            else:
                print("No hay suficientes datos procesados para calcular metricas estadisticas (se requiere n>1).")
        else:
            print("El archivo CSV esta vacio. Se requieren respuestas para el analisis.")
    except Exception as e:
        print(f"Error leyendo el archivo: {e}")


## 4. Amenazas a la Validez

Para garantizar la transparencia y el rigor metodológico, declaramos las siguientes amenazas a la validez de estos resultados:

- **Muestra por conveniencia**: Los participantes (n ≥ 15) fueron reclutados dentro del círculo cercano o accesible del equipo, no mediante un muestreo aleatorio probabilístico. Esto reduce la validez externa (generalización) de los hallazgos a toda la población potencial.
- **Sesgo de complacencia (Social Desirability Bias)**: Al conocer al equipo evaluador, los participantes podrían haber tendido a puntuar el sistema de forma más positiva para no "perjudicar" el trabajo del grupo.
- **Entorno de prueba controlado**: Las sesiones de *onboarding* se realizaron bajo una supervisión específica (presencial o videollamada), lo que no refleja un escenario de uso completamente orgánico y desatendido.
